In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/__huggingface_repos__.json
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/config.json
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/trainer_state.json
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/training_args.bin
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/scaler.pt
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/scheduler.pt
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/model.safetensors
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/optimizer.pt
/kaggle/input/datasets/binoyjose/model6-deberta-pairwise/deberta_checkpoints/checkpoint-2500/rng_state.pth
/kaggle/input/datasets/binoyjose/model6-deberta-pairwis

# Milestone 5

Master the competition metric, optimise inference, and combine models for maximum performance. Introduce TTA.


Suggested Readings:

* PyTorch – Softmax Function
* Hugging Face – Transformer
* Ensemble Learning

Setup (Run Before Attempting Questions) : 

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

**Question 1:**

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?


(answer format : eg - A, probability of A)

In [2]:
!pip install -q transformers torch accelerate sentencepiece

In [3]:
# Load the libraries
# ------------------------------
import torch
import torch.nn.functional as F
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [4]:
# Load Dataset
#----------------

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train Shape :", train.shape)
print("Test Shape :", test.shape)
print("Sample Submission Shape :", sample.shape)

Train Shape : (2000, 8)
Test Shape : (500, 7)
Sample Submission Shape : (500, 2)


In [5]:
# load the model

deberta_model_name = "microsoft/deberta-v3-small"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_name)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_model_name,
    num_labels=5
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model.to(device)
deberta_model.eval()

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [7]:
# Convert One MCQ to Text
row = train.iloc[25]

text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

In [8]:
# Tokeinze
# -------------------------------

inputs = deberta_tokenizer(
    text,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt"
)

inputs = {k:v.to(device) for k,v in inputs.items()}

In [9]:
# Inference

with torch.no_grad():
    outputs = deberta_model(**inputs)

logits = outputs.logits

In [10]:
# softmax

probabilities = F.softmax(logits, dim=-1)

print(probabilities)

tensor([[0.1726, 0.1511, 0.2444, 0.1965, 0.2352]], device='cuda:0',
       dtype=torch.float16)


In [11]:
# Highest probability

labels = ['A','B','C','D','E']

pred = torch.argmax(probabilities, dim=1).item()

print("Predicted Option :", labels[pred])
print("Probability :", probabilities[0][pred].item())

Predicted Option : C
Probability : 0.244384765625


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [12]:
# Load Fine-tuned RoBERTa Model
# --------------------------------

roberta_model_name = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_model_name,
    num_labels=5
)

roberta_model.to(device)
roberta_model.eval()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [13]:
# RoBERTa Inference
# -------------------
inputs = roberta_tokenizer(
    text,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt"
)

inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    outputs = roberta_model(**inputs)

roberta_probs = F.softmax(outputs.logits, dim=-1)

pred = torch.argmax(roberta_probs, dim=1).item()

print(labels[pred])
print(roberta_probs[0][pred].item())

E
0.2346237152814865


In [14]:
# Average probabilities
ensemble_probs = (probabilities + roberta_probs) / 2

print(ensemble_probs)

tensor([[0.1637, 0.1760, 0.2363, 0.1889, 0.2349]], device='cuda:0')


In [15]:
# Hughest probability

labels = ['A', 'B', 'C', 'D', 'E']

best_idx = torch.argmax(ensemble_probs, dim=1).item()

print("Predicted Option :", labels[best_idx])
print("Probability :", ensemble_probs[0][best_idx].item())

Predicted Option : C
Probability : 0.23632574081420898


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

**Question 3:**

Which answer option is ranked first after weighted ensembling?

In [16]:
# Weighted Ensemble
# -------------------

# DeBERTa weight = 0.70
# RoBERTa weight = 0.30

weighted_probs = (0.7 * probabilities) + (0.3 * roberta_probs)

print(weighted_probs)

tensor([[0.1673, 0.1660, 0.2395, 0.1920, 0.2351]], device='cuda:0')


In [18]:
# Top predictions

labels = ['A', 'B', 'C', 'D', 'E']

best_idx = weighted_probs.argmax(dim=1).item()

print(f"Predicted Option: {labels[best_idx]}")
print(f"Probability: {weighted_probs[0, best_idx].item():.6f}")

Predicted Option: C
Probability: 0.239501


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

**Question 4:**

What is the Top-3 prediction string for row index 25?


Example : C A E

In [19]:
# Top-3 Prediction
# -----------------------------------

labels = ['A', 'B', 'C', 'D', 'E']

# Sort probabilities in descending order
sorted_indices = torch.argsort(weighted_probs[0], descending=True)

# Top-3 labels
top3 = [labels[i] for i in sorted_indices[:3]]

# Kaggle submission format
prediction = " ".join(top3)

print("Top-3 Prediction:", prediction)

Top-3 Prediction: C E D


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

**Question 5:**

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [23]:
import pandas as pd
import torch
import torch.nn.functional as F


labels = ['A', 'B', 'C', 'D', 'E']

predictions = []

for _, row in test.iterrows():

    text = f"""
    Question:
    {row['prompt']}

    A. {row['A']}
    B. {row['B']}
    C. {row['C']}
    D. {row['D']}
    E. {row['E']}
    """

    # ----- DeBERTa -----
    inputs = deberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=-1)

    # ----- RoBERTa -----
    inputs = roberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=-1)

    # Weighted ensemble
    probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    top3 = torch.argsort(probs[0], descending=True)[:3]

    prediction = " ".join(labels[i] for i in top3)

    predictions.append(prediction)

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

   id prediction
0   1      C E D
1   2      D B E
2   3      C E D
3   4      C D B
4   5      C D B


In [24]:
print(len(submission))

500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

**Question 6:**

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [26]:
import pandas as pd
import torch
import torch.nn.functional as F

labels = ['A', 'B', 'C', 'D', 'E']

test_subset = test.iloc[:50]

changed = 0

for _, row in test_subset.iterrows():

    # -----------------------------
    # Original Prompt
    # -----------------------------
    original_text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # -----------------------------
    # Instruction-Augmented Prompt
    # -----------------------------
    augmented_text = f"""
Answer the following multiple-choice question carefully:

Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # ========= Original ==========
    inputs = deberta_tokenizer(
        original_text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits_original = deberta_model(**inputs).logits

    probs_original = F.softmax(logits_original, dim=-1)

    # ========= Augmented ==========
    inputs = deberta_tokenizer(
        augmented_text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits_augmented = deberta_model(**inputs).logits

    probs_augmented = F.softmax(logits_augmented, dim=-1)

    # ========= TTA ==========
    probs_tta = (probs_original + probs_augmented) / 2

    pred_original = probs_original.argmax(dim=1).item()
    pred_tta = probs_tta.argmax(dim=1).item()

    if pred_original != pred_tta:
        changed += 1

print("Number of changed predictions:", changed)

Number of changed predictions: 33


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

**Question 7:**

How many rows have different Top-1 predictions?

In [28]:
import torch
import torch.nn.functional as F

labels = ['A', 'B', 'C', 'D', 'E']

test_subset = test.iloc[:100]

different = 0

for _, row in test_subset.iterrows():

    text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # ------------------------
    # DeBERTa
    # ------------------------
    inputs = deberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=-1)

    # ------------------------
    # RoBERTa
    # ------------------------
    inputs = roberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=-1)

    # ------------------------
    # Weighted Ensemble
    # ------------------------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    deberta_pred = deberta_probs.argmax(dim=1).item()
    ensemble_pred = ensemble_probs.argmax(dim=1).item()

    if deberta_pred != ensemble_pred:
        different += 1

print("Different Top-1 Predictions:", different)

Different Top-1 Predictions: 13


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

**Question 8:**

How many rows have a positive confidence gain (greater than 0)?

In [29]:
import torch
import torch.nn.functional as F

labels = ['A', 'B', 'C', 'D', 'E']

test_subset = test.iloc[:100]

positive_gain = 0

for _, row in test_subset.iterrows():

    text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # -------------------------
    # DeBERTa
    # -------------------------
    inputs = deberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=-1)

    # -------------------------
    # RoBERTa
    # -------------------------
    inputs = roberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=-1)

    # -------------------------
    # Weighted Ensemble
    # -------------------------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Highest confidence
    deberta_confidence = deberta_probs.max().item()
    ensemble_confidence = ensemble_probs.max().item()

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_gain += 1

print("Rows with Positive Confidence Gain:", positive_gain)

Rows with Positive Confidence Gain: 17


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

**Question 9:**

How many rows have at least one change in their ordered Top-3 ranking after ensembling?


Examples:

A C D vs. A D C

In [31]:
import torch
import torch.nn.functional as F

labels = ['A', 'B', 'C', 'D', 'E']

test_subset = test.iloc[:100]

changed_top3 = 0

for _, row in test_subset.iterrows():

    text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # -----------------------
    # DeBERTa
    # -----------------------
    inputs = deberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=-1)

    # -----------------------
    # RoBERTa
    # -----------------------
    inputs = roberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=-1)

    # -----------------------
    # Weighted Ensemble
    # -----------------------
    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    # Top-3 indices
    deberta_top3 = torch.argsort(deberta_probs[0], descending=True)[:3]
    ensemble_top3 = torch.argsort(ensemble_probs[0], descending=True)[:3]

    # Convert to option labels
    deberta_prediction = [labels[i] for i in deberta_top3]
    ensemble_prediction = [labels[i] for i in ensemble_top3]

    # Count if ordered Top-3 differs
    if deberta_prediction != ensemble_prediction:
        changed_top3 += 1

print("Rows with changed Top-3 ranking:", changed_top3)

Rows with changed Top-3 ranking: 53


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

**Question 10:**


What is the final MAP@3 score? 

(Round to 4 decimal places.)

In [33]:
from sklearn.model_selection import train_test_split

train_split, valid_split = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [34]:
# Creating MAP@3 Function

# Function to calculate Average Precision at K (AP@K) for a single prediction
def apk(actual, predicted, k=3):

    if len(predicted) > k:                   # Keep only the top-k predictions
        predicted = predicted[:k]            

    for i, p in enumerate(predicted):        # Iterate through the predicted labels
        if p == actual:                      # If the correct answer is found, return the score
            return 1 / (i + 1)

    return 0                                 # Return 0 if the correct answer is not in the top-k predictions


# Function to calculate Mean Average Precision at K (MAP@K)
def mapk(actuals, predictions, k=3):
    
    # Compute the average AP@K score over the entire dataset
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

In [35]:
import torch
import torch.nn.functional as F
import numpy as np

labels = ['A', 'B', 'C', 'D', 'E']

scores = []

validation_subset = valid_split.iloc[:100]

for _, row in validation_subset.iterrows():

    text = f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # ---------------------
    # DeBERTa
    # ---------------------
    inputs = deberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        deberta_logits = deberta_model(**inputs).logits

    deberta_probs = F.softmax(deberta_logits, dim=-1)

    # ---------------------
    # RoBERTa
    # ---------------------
    inputs = roberta_tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        roberta_logits = roberta_model(**inputs).logits

    roberta_probs = F.softmax(roberta_logits, dim=-1)

    # ---------------------
    # Weighted Ensemble
    # ---------------------
    probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    top3 = torch.argsort(probs[0], descending=True)[:3]

    prediction = [labels[i] for i in top3]

    actual = row["answer"]

    scores.append(mapk(actual, prediction))

In [36]:
map3 = np.mean(scores)

print(f"MAP@3 = {map3:.4f}")

MAP@3 = 0.2000
